# Aave V3.1 — data transformation

**Step 1: load & display.** Reads the *latest* versioned CSV per table from
`query_result_data/` and displays it. The raw extracts are treated as
**read-only** — nothing here writes back to that folder.

All loading logic lives in `transform.py` (libraries it imports: `pandas`,
stdlib `re`/`pathlib`, and `data_validation` for the shared query-id → label map).
Edit `TABLE` to choose which table to display; `PREVIEW_ROWS` sets the preview size.

In [ ]:
# Cell 1 — imports + notebook-level settings
import pandas as pd
from IPython.display import display
import transform as tf
import data_validation as dv
import adv_validation as adv

PREVIEW_ROWS = 10  # rows shown in the inline preview (notebook-only setting)
# Cell 2 — see the available tables (label <- latest file)
for label, filename in sorted(tf.list_tables().items()):
    print(f"{label:30s} <- {filename}")

In [ ]:
# Cell 3 — load one table (read-only) and display it
TABLE = "supply_withdraw"  # <-- pick a label from Cell 2 (or pass a CSV path)

df_0_supply_withdraw = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_supply_withdraw.shape[0]} rows x {df_0_supply_withdraw.shape[1]} cols")
display(df_0_supply_withdraw.dtypes.to_frame("dtype"))
display(df_0_supply_withdraw.head(PREVIEW_ROWS))

In [ ]:
#7711171
# df_1_supply_withdraw = tf.load_table("query_7711171")
# print(f"{TABLE}: {df_1_supply_withdraw.shape[0]} rows x {df_1_supply_withdraw.shape[1]} cols")
# display(df_1_supply_withdraw.dtypes.to_frame("dtype"))
# display(df_1_supply_withdraw.head(PREVIEW_ROWS))
df_0_supply_withdraw.drop(["net_supply_flow_raw", "latest_supply_block", "latest_withdraw_block"], axis=1, inplace=True)
# df_1_supply_withdraw.drop(["metric", "unit"], axis=1, inplace=True)
# df_1_supply_withdraw = df_1_supply_withdraw.drop(df_1_supply_withdraw.index[[0, 1]])

#query_7798226 =  7714873
df_oracle_price_usd_eth = tf.load_table("oracle_price_usd_eth_weth_6h")
print(f"{TABLE}: {df_oracle_price_usd_eth.shape[0]} rows x {df_oracle_price_usd_eth.shape[1]} cols")
display(df_oracle_price_usd_eth.dtypes.to_frame("dtype"))
df_oracle_price_usd_eth


In [ ]:

stats = adv.statistical_validation(df_oracle_price_usd_eth, columns=["avg_price_usd", "avg_price_eth"],save=False)
print(stats[["column", "n_null", "null_pct", "n_zero", "zero_pct"]])

In [ ]:
# Cell — divide raw amount columns by 10**decimals -> real token units
# decimals come from the reference table (df_1); raw_token_amount rows only.
df_0_supply_withdraw_scaled = tf.scale_by_decimals(df_0_supply_withdraw, df_oracle_price_usd_eth, drop_raw=True)

print(f"raw cols scaled: {tf.raw_amount_columns(df_0_supply_withdraw)}")
display(df_0_supply_withdraw_scaled.head(PREVIEW_ROWS))
# Cell — validation checks on the scaled amount columns of df_0 (after scale_by_decimals)
import adv_validation as av

# the "_raw" amount columns became these scaled token-unit columns (drop_raw=True)
amount_cols = [c[: -len(tf.RAW_SUFFIX)] for c in tf.raw_amount_columns(df_0_supply_withdraw)]
print("amount columns:", amount_cols)

# for each amount column: negative-value check, range check, deviation score
# (plot=False -> values only, no plots)
for col in amount_cols:
    print(f"\n===== {col} =====")
    neg = av.negative_value_check(df_0_supply_withdraw_scaled, col, plot=False)   # any negatives? how many?
    # rng = av.range_check(df_0_supply_withdraw_scaled, col, plot=False)            # observed value range
    # dev = av.deviation_score(df_0_supply_withdraw_scaled, col, plot=False)        # per-row score in [-1, 1]

    print(f"negatives : {neg['n_negative']} of {neg['n_checked']} ({neg['negative_pct']}%)")
    # print(f"range     : min={rng['observed_min']}  max={rng['observed_max']}")
    # print(f"deviation : min={dev['score_min']:.3f}  max={dev['score_max']:.3f}  mean={dev['score_mean']:.3f}")

In [ ]:
amount_cols = [
    "supply_amount",
    "withdrawal_amount",
]
df_3_supply_withdraw = tf.multiply_by_price(df_0_supply_withdraw_scaled, df_oracle_price_usd_eth, amount_cols)
df_3_supply_withdraw

In [ ]:
amount_cols = [
    "supply_tx_count",
    "withdrawal_tx_count",
    "unique_suppliers",
    "unique_withdraw_users",
    "supply_amount_value_usd",
    "supply_amount_value_eth",
    "withdrawal_amount_value_usd",
    "withdrawal_amount_value_eth",
]

df_4_supply_withdraw = tf.aggregate_by_time_bucket(df_3_supply_withdraw, "time_bucket", amount_cols, agg_func='sum')
df_4_supply_withdraw

In [ ]:

# the "_raw" amount columns became these scaled token-unit columns (drop_raw=True)
amount_cols = [
    "supply_tx_count",
    "withdrawal_tx_count",
    "unique_suppliers",
    "unique_withdraw_users",
    "supply_amount_value_usd",
    "supply_amount_value_eth",
    "withdrawal_amount_value_usd",
    "withdrawal_amount_value_eth",
]
print("amount columns:", amount_cols)

# for each amount column: negative-value check, range check, deviation score
# (plot=False -> values only, no plots)
for col in amount_cols:
    print(f"\n===== {col} =====")


    neg = adv.negative_value_check(df_4_supply_withdraw, col, plot=False)
    # rng = av.range_check(df_4_supply_withdraw, col, plot=False)
    # dev = av.deviation_score(df_4_supply_withdraw, col, plot=False)

    print(f"negatives : {neg['n_negative']} of {neg['n_checked']} ({neg['negative_pct']}%)")
    # print(f"range     : min={rng['observed_min']}  max={rng['observed_max']}")
    # print(f"deviation : min={dev['score_min']:.3f}  max={dev['score_max']:.3f}  mean={dev['score_mean']:.3f}")


In [ ]:
stats = adv.statistical_validation(df_4_supply_withdraw, amount_cols, save=False)

print(stats[["column", 
             "null_pct", "n_zero", "zero_pct", "negative_pct",
    "n_sentinel", "mean", "std", "cv",
    # "min", "p01", "p05", "q25", "median", "q75", 
    # "p95", "p99", #"max",
    # "iqr", "mad", "skewness", "excess_kurtosis",
    # "n_outliers_iqr", "outlier_iqr_pct", "n_outliers_mad", "outlier_mad_pct",
    "heavy_tailed", "success",]])

In [ ]:
# load one table (read-only) and display it
TABLE = "borrow_repay"  # <-- pick a label from Cell 2 (or pass a CSV path)

df_0_borrow = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_borrow.shape[0]} rows x {df_0_borrow.shape[1]} cols")
display(df_0_borrow.dtypes.to_frame("dtype"))
display(df_0_borrow.head(PREVIEW_ROWS))

In [ ]:
# df_1 = tf.load_table("query_7711265")
# print(f"{TABLE}: {df_1.shape[0]} rows x {df_1.shape[1]} cols")
# display(df_1.dtypes.to_frame("dtype"))
# display(df_1.head(PREVIEW_ROWS))

df_0_borrow["last_borrow_rate"] = df_0_borrow["last_borrow_rate"].astype(float) / 10**27
df_0_borrow

In [ ]:
#oracle_price_usd_eth_weth_6h =  7714873
df_oracle_price_usd_eth = tf.load_table("oracle_price_usd_eth_weth_6h")
print(f"{TABLE}: {df_oracle_price_usd_eth.shape[0]} rows x {df_oracle_price_usd_eth.shape[1]} cols")
display(df_oracle_price_usd_eth.dtypes.to_frame("dtype"))
display(df_oracle_price_usd_eth.head(PREVIEW_ROWS))

In [ ]:
df_0_borrow.drop(["net_debt_flow","latest_borrow_block", "latest_repay_block"], axis=1, inplace=True)
DF_common = df_0_borrow[["time_bucket", "asset", "asset_symbol", "last_borrow_rate"]].copy()
df_0_borrow.drop(["last_borrow_rate"], axis=1, inplace=True)
df_0_borrow

In [ ]:
# Cell — divide borrow_repay raw amount columns by 10**decimals -> real token units
# decimals come from df_1_borrow (the oracle_price table's `decimals` column), per asset.
# df_2_borrow = tf.scale_by_decimals(df_0_borrow, df_1_borrow)

# print(f"raw cols scaled: {tf.raw_amount_columns(df_0_borrow)}")
# display(df_2_borrow.head(PREVIEW_ROWS))

# the "_raw" amount columns became these scaled token-unit columns (drop_raw=True)
# amount_cols = [c[: -len(tf.RAW_SUFFIX)] for c in tf.raw_amount_columns(df_0_borrow)]
# print("amount columns:", amount_cols)

# for each amount column: negative-value check, range check, deviation score
# (plot=False -> values only, no plots)

amount_cols = [
    "borrow_amount",
    "repay_amount",
]
for col in amount_cols:
    print(f"\n===== {col} =====")
    neg = av.negative_value_check(df_0_borrow, col, plot=False)   # any negatives? how many?
    # rng = av.range_check(df_2_borrow, col, plot=False)            # observed value range
    # dev = av.deviation_score(df_2_borrow, col, plot=False)        # per-row score in [-1, 1]

    print(f"negatives : {neg['n_negative']} of {neg['n_checked']} ({neg['negative_pct']}%)")
    # print(f"range     : min={rng['observed_min']}  max={rng['observed_max']}")
    # print(f"deviation : min={dev['score_min']:.3f}  max={dev['score_max']:.3f}  mean={dev['score_mean']:.3f}")



In [ ]:


df_2_borrow = tf.multiply_by_price(df_0_borrow, df_oracle_price_usd_eth, amount_cols)
df_2_borrow



In [ ]:
borrow_agg_cols = [
    "borrow_tx_count",
    "repay_tx_count",
    # "stable_borrow_tx_count",
    "variable_borrow_tx_count",
    "unique_borrowers",
    "unique_repayers",
    "borrow_amount_value_usd",
    "borrow_amount_value_eth",
    "repay_amount_value_usd",
    "repay_amount_value_eth",
]

df_3_borrow = tf.aggregate_by_time_bucket(df_2_borrow, "time_bucket", borrow_agg_cols, agg_func='sum')
# df_3_borrow.drop(["stable_borrow_tx_count"], axis=1, inplace=True )

df_3_borrow

In [ ]:
stats_borr = adv.statistical_validation(df_3_borrow, borrow_agg_cols, save=False)

# features -> rows, metrics -> columns
metric_cols = ["mean", "median", "std", "cv", "p95", "p99"]
stats = stats_borr.set_index("column")[metric_cols]

print(stats)

In [ ]:
TABLE = "reserve_state_rates"  

df_0_reserve_state_rates = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_reserve_state_rates.shape[0]} rows x {df_0_reserve_state_rates.shape[1]} cols")
display(df_0_reserve_state_rates.dtypes.to_frame("dtype"))
display(df_0_reserve_state_rates.head(PREVIEW_ROWS))

In [ ]:
reserve_cols = [
    "liquidity_rate",
    "stable_borrow_rate",
    "variable_borrow_rate",
    "liquidity_index",
    "variable_borrow_index",
]

# for col in reserve_cols:
#     df_0_reserve_state_rates[col] = df_0_reserve_state_rates[col].astype(float) / 10**27

DF_common_1 = DF_common.merge(
    df_0_reserve_state_rates,
      on=["time_bucket", "asset"], how="left", suffixes=("_x", "_y"))

DF_common_1.drop(["stable_borrow_rate"], axis=1, inplace=True)
DF_common_1

In [ ]:
common_cols = [
    "last_borrow_rate",
    "liquidity_rate",
    "variable_borrow_rate",
    "liquidity_index",
    "variable_borrow_index",
]

stats = pd.DataFrame({
    "mean": DF_common_1[common_cols].mean(),
    "median": DF_common_1[common_cols].median(),
    "std": DF_common_1[common_cols].std(),
    "variance": DF_common_1[common_cols].var(),
    "cv": DF_common_1[common_cols].std() / DF_common_1[common_cols].mean(),
    "p95": DF_common_1[common_cols].quantile(0.95),
    "p99": DF_common_1[common_cols].quantile(0.99)
})

print(stats)

In [ ]:
TABLE = "query_7804264" 

df_0_reserve_config = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_reserve_config.shape[0]} rows x {df_0_reserve_config.shape[1]} cols")
display(df_0_reserve_config.dtypes.to_frame("dtype"))
display(df_0_reserve_config.head(PREVIEW_ROWS))
summary = pd.DataFrame({
    "null_count": df_0_reserve_config.isna().sum(),
    "null_pct": df_0_reserve_config.isna().mean() * 100,
})
print(summary)

# reserve config results is broken, not enough nor consistent data with very high percentage of null values, so we will not use it for now.
# still compromised. the null values has been replaced with 0 and 1, but the data is not reliable enough to be used confidently in the analysis.

In [ ]:
df_oracle_price_usd_eth

In [ ]:
TABLE = "query_7711290"

df_1_reserve_config = tf.load_table(TABLE)
print(f"{TABLE}: {df_1_reserve_config.shape[0]} rows x {df_1_reserve_config.shape[1]} cols")
display(df_1_reserve_config.dtypes.to_frame("dtype"))
# display(df_1_reserve_config.head(PREVIEW_ROWS))
df_1_reserve_config

In [ ]:
# reserve_config is pre-normalized in normalize.ipynb — scaling removed here
df_2_reserve_config = df_0_reserve_config
df_2_reserve_config

In [ ]:
config_amount_cols = [
    "supply_cap",
    "borrow_cap",
    "debt_ceiling",
]

df_3_reserve_config = tf.multiply_by_price(df_2_reserve_config, df_oracle_price_usd_eth, config_amount_cols)
df_3_reserve_config




In [ ]:
config_amount_cols = [
    "supply_cap_value_usd",
    "supply_cap_value_eth",
    "borrow_cap_value_usd",
    "borrow_cap_value_eth",
    "debt_ceiling_value_usd",
    "debt_ceiling_value_eth",
]

df_4_reserve_config = (
    df_3_reserve_config
    [["time_bucket"] + config_amount_cols]
)

df_4_reserve_config_final = tf.aggregate_by_time_bucket(
    df_4_reserve_config,
    "time_bucket",
    config_amount_cols,
    agg_func="sum"
)

df_4_reserve_config_final

In [ ]:
df_3_reserve_config.drop(["asset_symbol", "supply_cap", "borrow_cap", "debt_ceiling", "config_event_count"], axis=1, inplace=True)
df_3_reserve_config = dv.canonicalize_keys(df_3_reserve_config)   # canonical time_bucket so the merge matches the panel
DF_common_2 = DF_common_1.merge(
    df_3_reserve_config,
      on=["time_bucket", "asset"], how="left", suffixes=("_x", "_y"))

DF_common_2

In [ ]:
TABLE = "liquidation"

df_0_liq = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_liq.shape[0]} rows x {df_0_liq.shape[1]} cols")
display(df_0_liq.dtypes.to_frame("dtype"))
# display(df_0_liq.head(PREVIEW_ROWS))
df_0_liq

In [ ]:
TABLE = "query_7711290" 

df_1_liq = tf.load_table(TABLE)
print(f"{TABLE}: {df_1_liq.shape[0]} rows x {df_1_liq.shape[1]} cols")
display(df_1_liq.dtypes.to_frame("dtype"))
display(df_1_liq.head(PREVIEW_ROWS))

In [ ]:
df_2_liq = tf.scale_by_decimals(df_0_liq, df_oracle_price_usd_eth, drop_raw=True)
df_3_liq = tf.multiply_by_price(df_2_liq, df_oracle_price_usd_eth, ["liquidated_collateral", "liquidation_debt_covered"])
df_3_liq


In [ ]:
df_3_liq.drop(["latest_liquidation_block"], axis=1, inplace=True)
liq_agg_cols = [
    "as_collateral_tx_count",
    "as_debt_tx_count",
    "liquidation_tx_count",
    "unique_liquidated_users",
    "unique_liquidators",
    "liquidated_collateral_value_usd",
    "liquidated_collateral_value_eth",
    "liquidation_debt_covered_value_usd",
    "liquidation_debt_covered_value_eth",
]
df_4_liq = tf.aggregate_by_time_bucket(df_3_liq, "time_bucket", liq_agg_cols, agg_func='sum')
df_4_liq

In [ ]:
TABLE = "collateral_toggle"

df_0_collateral = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_collateral.shape[0]} rows x {df_0_collateral.shape[1]} cols")
display(df_0_collateral.dtypes.to_frame("dtype"))
display(df_0_collateral.head(PREVIEW_ROWS))

In [ ]:
collateral_agg_cols = [
    "collateral_enabled_count",
    "collateral_disabled_count",
    "unique_collateral_enable_users",
    "unique_collateral_disable_users"
]

df_1_collateral = tf.aggregate_by_time_bucket(df_0_collateral, "time_bucket", collateral_agg_cols, agg_func='sum')
df_1_collateral

In [ ]:
TABLE = "flashloan"

df_0_flashloan = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_flashloan.shape[0]} rows x {df_0_flashloan.shape[1]} cols")
display(df_0_flashloan.dtypes.to_frame("dtype"))
display(df_0_flashloan.head(PREVIEW_ROWS))

In [ ]:
TABLE = "query_7711298"

df_2_flashloan = tf.load_table(TABLE)
print(f"{TABLE}: {df_2_flashloan.shape[0]} rows x {df_2_flashloan.shape[1]} cols")
# display(df_2_flashloan.dtypes.to_frame("dtype"))
# display(df_2_flashloan.head(PREVIEW_ROWS))

# split the reference frame by its `metric` column
df_2_flashloan_amount  = df_2_flashloan[df_2_flashloan["metric"] == "flashloan_amount_raw"].copy()
df_2_flashloan_premium = df_2_flashloan[df_2_flashloan["metric"] == "flashloan_premium_raw"].copy()

display(df_2_flashloan_amount.head(PREVIEW_ROWS))
display(df_2_flashloan_premium.head(PREVIEW_ROWS))

In [ ]:
flashloan_agg_cols = [
    "flashloan_tx_count",
    "unique_flashloan_initiators",
    "no_open_debt_flashloan_tx_count",
    "variable_flashloan_tx_count",
]

df_1_flashloan = tf.aggregate_by_time_bucket(df_0_flashloan, "time_bucket", flashloan_agg_cols, agg_func='sum')
df_1_flashloan

In [ ]:
df_2_flashloan_scaled = tf.scale_by_decimals(df_0_flashloan, df_2_flashloan_amount, drop_raw=True)
df_2_flashloan_scaled.drop(flashloan_agg_cols, axis=1, inplace=True)
df_2_flashloan_scaled.drop(["stable_flashloan_tx_count", "latest_flashloan_block"], axis=1, inplace=True)
df_2_flashloan_scaled_1 = tf.multiply_by_price(df_2_flashloan_scaled, df_oracle_price_usd_eth, ["flashloan_amount", "flashloan_premium"])
# df_3_borrow = tf.aggregate_by_time_bucket(df_2_borrow, "time_bucket", borrow_agg_cols, agg_func='sum')

stats_flashloan = [
    "flashloan_amount_value_usd",
    "flashloan_amount_value_eth",
    "flashloan_premium_value_usd",
    "flashloan_premium_value_eth",
]

df_2_flashloan_scaled_1



In [ ]:
df_2_flashloan_scaled_2 = tf.aggregate_by_time_bucket(df_2_flashloan_scaled_1, "time_bucket", stats_flashloan , agg_func='sum')
df_2_flashloan_scaled_2


In [ ]:
DF_flashloan_common = df_2_flashloan_scaled_2.merge(
    df_1_flashloan,
      on=["time_bucket"], how="left", suffixes=("_x", "_y"))

DF_flashloan_common

In [ ]:
TABLE = "user_account"

df_0_user_account = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_user_account.shape[0]} rows x {df_0_user_account.shape[1]} cols")
display(df_0_user_account.dtypes.to_frame("dtype"))
display(df_0_user_account.head(PREVIEW_ROWS))
df_0_user_account.drop(["min_health_factor", "max_health_factor"], axis=1, inplace=True)

In [ ]:
stats_user = [
    df_0_user_account["sampled_user_count"].mean(), 
    df_0_user_account["sampled_user_count"].median(),
    # df_0_user_account["sampled_user_count"].std(),
    # df_0_user_account["sampled_user_count"].var(),
    df_0_user_account["account_data_call_count"].mean(),
    df_0_user_account["account_data_call_count"].median(),
#   df_0_user_account["account_data_call_count"].std(),
#   df_0_user_account["account_data_call_count"].var()
]

print(stats_user)

In [ ]:
TABLE = "query_7711304"

df_0_user_account_agg = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_user_account_agg.shape[0]} rows x {df_0_user_account_agg.shape[1]} cols")
display(df_0_user_account_agg.dtypes.to_frame("dtype"))
# display(df_0_user_account_agg.head(PREVIEW_ROWS))

df_0_user_account_agg_1 = df_0_user_account_agg.drop(df_0_user_account_agg.index[[0, 8]])
df_0_user_account_agg_1


In [ ]:
metric_cols = dict(zip(
    df_0_user_account_agg_1["metric"],
    df_0_user_account_agg_1["decimals"]
))

    
out = df_0_user_account.copy()

for col, dec in metric_cols.items():
    if col not in out.columns:
        continue  # skip metrics that aren't columns in this frame
    out[col] = out[col].astype("float64") / (10 ** int(dec))
df_0_user_account_agg_scaled = out
df_0_user_account_agg_scaled

In [ ]:
# df_4_supply_withdraw
# df_3_borrow
# DF_common_1
# df_4_liq
# df_1_collateral
# DF_flashloan_common
# df_0_user_account_agg_scaled

DF_common_final = df_4_supply_withdraw.merge(
    df_3_borrow, on=["time_bucket"], how="left", suffixes=("", "_y")).merge(
    df_4_liq, on=["time_bucket"], how="left", suffixes=("", "_liq")).merge(
    df_1_collateral, on=["time_bucket"], how="left", suffixes=("", "_collateral")).merge(
    DF_flashloan_common, on=["time_bucket"], how="left", suffixes=("", "_flashloan")).merge(
    df_0_user_account_agg_scaled, on=["time_bucket"], how="left", suffixes=("", "_useragg"))

display(DF_common_final.head(PREVIEW_ROWS))
DF_common_final
display(DF_common_final.dtypes.to_frame("dtype"))
display(DF_common_1.head(PREVIEW_ROWS))


In [ ]:
from pathlib import Path

OUT_DIR = Path("transformed_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DF_common_final.to_csv(OUT_DIR / "DF_common_final.csv", index=False)
DF_common_1.to_csv(OUT_DIR / "DF_common_1.csv", index=False)

OUT_DIR_1 = Path("processed_data")

processed_tables = [
    df_4_supply_withdraw,
    df_3_borrow,
    DF_common_1,
    df_4_liq,
    df_1_collateral,
    DF_flashloan_common,
    df_0_user_account_agg_scaled,
    ]

for i, df in enumerate(processed_tables):
    df.to_csv(OUT_DIR_1 / f"table_{i}.csv", index=False)
    print(f"wrote {len(df)} rows to {OUT_DIR_1}/")



print(f"wrote {len(DF_common_final)} + {len(DF_common_1)} rows to {OUT_DIR}/")